# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/HassanKh4lil/ML-WEEK-1/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

In [6]:
%pip -q install duckdb
import duckdb

con = duckdb.connect()
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get('HF_TOKEN')
except Exception:
    import os
    HF_TOKEN = os.environ.get('HF_TOKEN')

con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

FACT = "read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet')"
DIMC = "read_parquet('hf://datasets/FlyRank/internship-warehouse/dim_content.parquet')"
DIMCL = "read_parquet('hf://datasets/FlyRank/internship-warehouse/dim_clients.parquet')"

print(con.sql(f"DESCRIBE SELECT * FROM {FACT} LIMIT 0").df())

Enter your Hugging Face READ token: ··········
                 column_name column_type null   key default extra
0                report_date        DATE  YES  None    None  None
1             client_hash_id     VARCHAR  YES  None    None  None
2            content_hash_id     VARCHAR  YES  None    None  None
3             client_has_gsc     BOOLEAN  YES  None    None  None
4             client_has_ga4     BOOLEAN  YES  None    None  None
5         gsc_data_available     BOOLEAN  YES  None    None  None
6         ga4_data_available     BOOLEAN  YES  None    None  None
7            gsc_impressions      BIGINT  YES  None    None  None
8                 gsc_clicks      BIGINT  YES  None    None  None
9           gsc_sum_position      BIGINT  YES  None    None  None
10          gsc_avg_position      DOUBLE  YES  None    None  None
11             ga4_pageviews      BIGINT  YES  None    None  None
12              ga4_sessions      BIGINT  YES  None    None  None
13                 ga4_users 

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

In [8]:
DECISION_WINDOW_START = "DATE '2026-03-01'"
DECISION_WINDOW_END   = "DATE '2026-03-30'"

feature_frame_sql = f"""
WITH windowed AS (
    SELECT
        f.content_hash_id,
        f.client_hash_id,
        SUM(f.gsc_impressions)                AS impressions_30d,
        SUM(f.gsc_clicks)                      AS clicks_30d,
        AVG(NULLIF(f.gsc_avg_position, 0))     AS avg_position_30d,
        SUM(f.ga4_sessions)                    AS sessions_30d,
        BOOL_OR(f.ga4_data_available IS TRUE)  AS ga4_available_any
    FROM {FACT} f
    WHERE f.report_date BETWEEN {DECISION_WINDOW_START} AND {DECISION_WINDOW_END}
    GROUP BY 1, 2
)
SELECT
    w.content_hash_id,
    w.client_hash_id,
    w.impressions_30d,
    w.clicks_30d,
    ROUND(100.0 * w.clicks_30d / NULLIF(w.impressions_30d, 0), 2) AS ctr,
    w.avg_position_30d,
    CASE
        WHEN w.avg_position_30d IS NULL THEN 'no_data'
        WHEN w.avg_position_30d <= 3  THEN 'top_3'
        WHEN w.avg_position_30d <= 10 THEN 'page_1'
        WHEN w.avg_position_30d <= 20 THEN 'striking'
        WHEN w.avg_position_30d <= 50 THEN 'page_3_5'
        ELSE 'deep'
    END AS position_tier,
    w.sessions_30d,
    w.ga4_available_any,
    d.word_count,
    DATE_DIFF('day', d.content_created_date, {DECISION_WINDOW_END}) AS content_age_days
FROM windowed w
LEFT JOIN {DIMC} d ON d.content_hash_id = w.content_hash_id
WHERE w.impressions_30d >= 100
"""

feature_frame = con.sql(feature_frame_sql).df()
print(feature_frame.shape)
feature_frame.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

(100376, 11)


,content_hash_id,client_hash_id,impressions_30d,clicks_30d,ctr,avg_position_30d,position_tier,sessions_30d,ga4_available_any,word_count,content_age_days
0,content_2e6360ad20fd7107,client_62f4a7e64f5e0096,786.0,1.0,0.13,5.937241,page_1,NaN,False,2855,46
1,content_65c50dfe9d87a585,client_62f4a7e64f5e0096,3002.0,0.0,0.00,7.008173,page_1,NaN,False,2779,46
2,content_275b6f7f733016d4,client_62f4a7e64f5e0096,791.0,1.0,0.13,4.842544,page_1,NaN,False,3653,46
3,content_4dc944b7d0b65ecc,client_62f4a7e64f5e0096,134.0,0.0,0.00,5.012831,page_1,NaN,False,3698,46
4,content_92c381fbd361212e,client_62f4a7e64f5e0096,523.0,1.0,0.19,4.518788,page_1,NaN,False,3248,46


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [9]:
grain_check = con.sql(f"""
    SELECT report_date, client_hash_id, content_hash_id, COUNT(*) AS c
    FROM {FACT}
    GROUP BY 1, 2, 3
    HAVING COUNT(*) > 1
    LIMIT 5
""").df()
print("Duplicate grain rows found:", len(grain_check))
grain_check

span = con.sql(f"""
    SELECT
        COUNT(*)                          AS n_rows,
        MIN(report_date)                  AS min_date,
        MAX(report_date)                  AS max_date,
        COUNT(DISTINCT client_hash_id)    AS n_clients,
        COUNT(DISTINCT content_hash_id)   AS n_content_items
    FROM {FACT}
""").df()
span

availability = con.sql(f"""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(*) FILTER (WHERE ga4_data_available IS TRUE)  AS ga4_available_rows,
        COUNT(*) FILTER (WHERE ga4_data_available IS FALSE) AS ga4_unavailable_rows,
        COUNT(*) FILTER (WHERE ga4_data_available IS NULL)  AS ga4_null_rows
    FROM {FACT}
""").df()
availability['pct_survive_is_true'] = (
    100.0 * availability['ga4_available_rows'] / availability['total_rows']
).round(1)
availability

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Duplicate grain rows found: 0


,total_rows,ga4_available_rows,ga4_unavailable_rows,ga4_null_rows,pct_survive_is_true
0,9841378,413966,6408671,3018741,4.2


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

In [11]:
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

honest_features = ['impressions_30d', 'sessions_30d', 'word_count', 'content_age_days']

df = feature_frame.dropna(subset=['position_tier'] + honest_features).copy()

tier_median_ctr = df.groupby('position_tier')['ctr'].transform('median')
df['is_ctr_review_candidate'] = (df['ctr'] < 0.6 * tier_median_ctr).astype(int)

X_honest = pd.get_dummies(df[honest_features + ['position_tier']], columns=['position_tier'])
y = df['is_ctr_review_candidate']

Xtr, Xte, ytr, yte = train_test_split(X_honest, y, test_size=0.3, random_state=42, stratify=y)
honest_model = LogisticRegression(max_iter=1000).fit(Xtr, ytr)
honest_auc = roc_auc_score(yte, honest_model.predict_proba(Xte)[:, 1])
print(f"HONEST score (5 features, no leak): ROC AUC = {honest_auc:.3f}")

X_leak = X_honest.copy()
X_leak['ctr'] = df['ctr'].values

Xtr_l, Xte_l, ytr_l, yte_l = train_test_split(X_leak, y, test_size=0.3, random_state=42, stratify=y)
leaky_model = LogisticRegression(max_iter=1000).fit(Xtr_l, ytr_l)
leaky_auc = roc_auc_score(yte_l, leaky_model.predict_proba(Xte_l)[:, 1])
print(f"LEAKY score (ctr column added):     ROC AUC = {leaky_auc:.3f}")

del X_leak
print(f"\nKept number: honest ROC AUC = {honest_auc:.3f}. "
      f"The {leaky_auc - honest_auc:.3f}-point jump came entirely from feeding the model "
      f"the same number its label was derived from -- not from real signal.")

/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


HONEST score (5 features, no leak): ROC AUC = 0.690
LEAKY score (ctr column added):     ROC AUC = 1.000

Kept number: honest ROC AUC = 0.690. The 0.309-point jump came entirely from feeding the model the same number its label was derived from -- not from real signal.


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.